# v21 案A: 観測尤度を学習(ニューラル・マッチャー)

PFの GR距離尤度を、**2タワーSiamese**が出す学習コストマップ `Lmap[md_i, tvt_grid]` に置換。
- 水平GR窓(MD周辺) と typewell GR窓(候補tvt周辺) を各タワーで埋め込み → **内積でコストマップを1発生成**(GPU高速)。
- 学習: trainの真tvtを正解、他位置を負例に InfoNCE(PF微分不要)。
- PF: 粒子位置→grid index で `Lmap` を lookup(`v21_common._pf_core(lmap=...)`)。
- 評価: 素のr0 PF と overall / 硬い核 per-well を比較(ノイズ床±0.2、桁変化を見る)。

In [1]:
import sys, time, numpy as np, torch, torch.nn as nn
sys.path.insert(0, "v21")
import importlib, v21_common as C; importlib.reload(C)
torch.manual_seed(0); np.random.seed(0)
DEV = C.DEVICE; print("device:", DEV)

# ---- 窓・モデル設定 ----
HW   = 32          # 水平GR窓の半幅(MD行) → 長さ 2*HW+1
TWW  = 64          # typewell窓の半幅(tvtステップ)
TWS  = 0.5         # typewell窓の刻み(ft)
EMB  = 64          # 埋め込み次元
TEMP = 0.15        # InfoNCE/Lmap 温度
N_TRAIN_WELLS = 250  # matcher学習に使う坑井数(速度優先で部分使用)

ns = C.load_v9_ns()
p  = C.r0_params(ns, use_phys=False, n_seeds=8)
WELLS = C.list_train_wells()

device: cuda
[v9] exec 中(FI/DI fit を含む)...
PF parameter sets:
  pf_1: suffix=_1, N=600, seeds=32, jump=(0.002, 5.0)
  pf_2: suffix=_2, N=600, seeds=32, jump=(0.0, 10.0)
  pf_3: suffix=_3, N=600, seeds=32, jump=(0.002, 5.0)
[v21] r0 params: r0 N=500 seeds=8 phys=off


In [2]:
# ===== 全行データ(md, 処理GR, 真tvt, typewell)を読む =====
import pandas as pd
def load_full(wid):
    base = C.DATA_DIR / "train"
    hw = pd.read_csv(base / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(base / f"{wid}__typewell.csv").sort_values("TVT")
    if "TVT" not in hw.columns or hw["TVT"].isna().all(): return None
    tw_tvt = tw["TVT"].to_numpy(np.float64); tw_gr = tw["GR"].to_numpy(np.float64)
    if len(tw_tvt) < 5: return None
    _smooth = ns["_smooth_radius_values"]
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    tw_gr_pf = _smooth(tw_gr, tw_fb, p["PF_TW_GR_SMOOTH_R"]).astype(np.float64)
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    gr_proc = _smooth(gr_full, tw_fb, p["PF_HGR_SMOOTH_R"]).astype(np.float32)
    tvt = hw["TVT"].to_numpy(np.float64)
    ok = np.isfinite(tvt)
    return dict(gr=gr_proc, tvt=tvt, ok=ok, tw_tvt=tw_tvt, tw_gr=tw_gr_pf.astype(np.float32),
                gr_mean=float(np.nanmean(gr_proc)), gr_std=float(np.nanstd(gr_proc)+1e-6))

DB = []
for wid in WELLS[:N_TRAIN_WELLS]:
    d = load_full(wid)
    if d is not None and d["ok"].sum() > 100: DB.append(d)
print("matcher学習坑井:", len(DB))

def hwin(gr, i, gmean, gstd):
    L = 2*HW+1; a = i-HW; b = i+HW+1
    lo = max(0,a); hi = min(len(gr),b)
    out = np.full(L, gr[lo] if i<HW else gr[-1], np.float32)
    out[lo-a:hi-a] = gr[lo:hi]
    return (out - gmean)/gstd

def twin(tw_tvt, tw_gr, t, gmean, gstd):
    pos = t + np.arange(-TWW, TWW+1)*TWS
    g = np.interp(pos, tw_tvt, tw_gr).astype(np.float32)
    return (g - gmean)/gstd

# サンプル(well_idx, row) を作る(各坑井から最大160行)
SAMPLES = []
for wi, d in enumerate(DB):
    rows = np.where(d["ok"])[0]
    if len(rows) > 160: rows = rows[np.linspace(0, len(rows)-1, 160).astype(int)]
    for r in rows: SAMPLES.append((wi, r))
SAMPLES = np.array(SAMPLES); print("学習サンプル:", len(SAMPLES))

matcher学習坑井: 250
学習サンプル: 40000


In [3]:
# ===== 2タワー 1D-CNN =====
class Tower(nn.Module):
    def __init__(self, emb=EMB):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1,32,7,padding=3), nn.ReLU(), nn.Conv1d(32,32,5,padding=2), nn.ReLU(),
            nn.MaxPool1d(2), nn.Conv1d(32,64,5,padding=2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Linear(64, emb)
    def forward(self, x):              # x:(B,L)
        z = self.net(x.unsqueeze(1)).squeeze(-1)
        z = self.fc(z)
        return nn.functional.normalize(z, dim=-1)

enc_h = Tower().to(DEV); enc_t = Tower().to(DEV)
opt = torch.optim.Adam(list(enc_h.parameters())+list(enc_t.parameters()), lr=1e-3)
print("towers ready | params:", sum(x.numel() for x in enc_h.parameters())*2)

towers ready | params: 39744


In [5]:
# ===== 学習(InfoNCE: 同バッチ内+同坑井ハード負例)=====
EPOCHS = 8; BS = 256; HARDNEG = 4   # 各アンカーに同坑井のズレ位置を負例追加
def make_batch(idx):
    hs=[]; ts=[]; negs=[]
    for wi, r in SAMPLES[idx]:
        d = DB[wi]; t0 = d["tvt"][r]
        hs.append(hwin(d["gr"], r, d["gr_mean"], d["gr_std"]))
        ts.append(twin(d["tw_tvt"], d["tw_gr"], t0, d["gr_mean"], d["gr_std"]))
        for _ in range(HARDNEG):
            off = np.random.choice([-1,1])*np.random.uniform(8, 60)   # 8〜60ft ずらした偽位置
            negs.append(twin(d["tw_tvt"], d["tw_gr"], t0+off, d["gr_mean"], d["gr_std"]))
    H = torch.tensor(np.array(hs), device=DEV)
    T = torch.tensor(np.array(ts), device=DEV)
    Ng = torch.tensor(np.array(negs), device=DEV) if negs else None
    return H, T, Ng

for ep in range(EPOCHS):
    perm = np.random.permutation(len(SAMPLES)); tot=0; nb=0; t0=time.time()
    enc_h.train(); enc_t.train()
    for s in range(0, len(perm), BS):
        idx = perm[s:s+BS]
        H,T,Ng = make_batch(idx)
        eh = enc_h(H); et = enc_t(T)                       # (B,EMB)
        logits = eh @ et.t() / TEMP                        # (B,B) 対角=正
        if Ng is not None:
            en = enc_t(Ng).view(len(idx), HARDNEG, EMB)    # (B,HN,EMB)
            hardl = (eh.unsqueeze(1)*en).sum(-1)/TEMP      # (B,HN)
            logits = torch.cat([logits, hardl], 1)         # (B, B+HN)
        lab = torch.arange(len(idx), device=DEV)
        loss = nn.functional.cross_entropy(logits, lab)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item(); nb += 1
    print(f"ep{ep} loss={tot/nb:.4f} ({time.time()-t0:.1f}s)")
torch.save({"enc_h":enc_h.state_dict(),"enc_t":enc_t.state_dict()}, "matcher_A.pt")
print("saved v21/matcher_A.pt")

ep0 loss=4.6609 (3.5s)
ep1 loss=4.6113 (3.4s)
ep2 loss=4.5623 (3.3s)
ep3 loss=4.5236 (3.3s)
ep4 loss=4.4769 (3.3s)
ep5 loss=4.4288 (3.3s)
ep6 loss=4.3918 (3.3s)
ep7 loss=4.3535 (3.3s)
saved v21/matcher_A.pt


In [6]:
# ===== lmap_fn: 学習matcherでコストマップ Lmap[T_eval, Gl] を生成(GPU, 行列積1発) =====
@torch.no_grad()
def lmap_fn(wid, inp):
    enc_h.eval(); enc_t.eval()
    d = load_full(wid)                                     # 同じ正規化を使う
    if d is None:
        gmean, gstd = float(np.nanmean(inp["gr"])), float(np.nanstd(inp["gr"])+1e-6)
        tw_tvt, tw_gr = inp["tw_tvt"], inp["tw_gr"].astype(np.float32)
    else:
        gmean, gstd, tw_tvt, tw_gr = d["gr_mean"], d["gr_std"], d["tw_tvt"], d["tw_gr"]
    gre = inp["gr"]; T = len(gre)                          # eval の処理済GR
    Hs = np.stack([hwin(gre, i, gmean, gstd) for i in range(T)]).astype(np.float32)
    lgrid = inp["lgrid"]; G = len(lgrid)
    Ts = np.stack([twin(tw_tvt, tw_gr, t, gmean, gstd) for t in lgrid]).astype(np.float32)
    eh = enc_h(torch.tensor(Hs, device=DEV))               # (T,EMB)
    et = enc_t(torch.tensor(Ts, device=DEV))               # (G,EMB)
    scores = (eh @ et.t()) / TEMP                          # (T,G)
    logp = torch.log_softmax(scores, dim=1)                # 位置の対数確率(安定)
    return logp.float().cpu().numpy()

print("lmap_fn ready")

lmap_fn ready


In [7]:
# ===== 評価: 素のr0 PF vs 案A(学習尤度) =====
EVAL = WELLS[:40] + C.HARD_WELLS
EVAL = list(dict.fromkeys(EVAL))            # 重複除去
base = C.eval_wells(EVAL, ns, p, chunk=4)
A    = C.eval_wells(EVAL, ns, p, chunk=2, lmap_fn=lmap_fn)
print(f"\n=== overall: base={base['overall']:.3f}  A(lmap)={A['overall']:.3f}  d={A['overall']-base['overall']:+.3f} ===")
print(f"\n{'well':10s} {'base':>7s} {'A':>7s} {'d':>7s}  (hard)")
for w in base["perwell"]:
    if w not in A["perwell"]: continue
    b=base["perwell"][w]; a=A["perwell"][w]
    h=" <hard" if w in C.HARD_WELLS else ""
    if h or abs(a-b)>0.5:
        print(f"{w:10s} {b:7.2f} {a:7.2f} {a-b:+7.2f}{h}")
hb=[base["perwell"][w] for w in C.HARD_WELLS if w in base["perwell"]]
ha=[A["perwell"][w]    for w in C.HARD_WELLS if w in A["perwell"]]
print(f"\nhard mean: base={np.mean(hb):.2f}  A={np.mean(ha):.2f}")

[eval] wells=59 overall=16.891  (110.0s, cuda) 
[eval] wells=59 overall=26.724  (174.9s, cuda) A:lmap

=== overall: base=16.891  A(lmap)=26.724  d=+9.833 ===

well          base       A       d  (hard)
000d7d20      6.85   20.43  +13.57
00bbac68      4.87   21.86  +16.99
00e12e8b     12.25   11.20   -1.05
015fe0d2      5.23   11.74   +6.52
01869cd4     12.61   16.26   +3.65
01982c1d      5.75   14.65   +8.90
028d7b28     13.95    9.18   -4.78
02e7fe5a      4.11    8.66   +4.55
0390d174      2.33   13.09  +10.76
03a935ae      2.16   26.57  +24.41
044af7d1      2.49   18.00  +15.52
0498acab      2.56    8.95   +6.40
052d64df      4.76    7.67   +2.91
05948241      1.28   11.07   +9.79
059c8f24      5.29   12.65   +7.36
05a0ee4d     11.05    8.23   -2.81
060ab2b8      3.28   18.97  +15.70
06df5958      4.35    2.89   -1.45
071d7b45     10.51    4.53   -5.98
0849b4e2      3.57   15.58  +12.01
084d66f9      2.47   29.98  +27.51
08fecf00      8.12   10.44   +2.33
09053135     12.98   15.09  